# Update pickle file only

In [2]:
import datetime as dt
import pandas as pd
import numpy as np
import re
import requests
import pymssql
import pickle
mssql = {'server':'192.168.61.119:7622', 'database':'mars', 'user':'BAReporting', 'password':'KeHeCReme8he'}

In [3]:
!pwd

/ReportingRoot/JupyterRoot/reports/TakeAway_Merchant_Offline_Alert_Report


In [4]:
try:
    with open(r"offed_pois.obj", "rb") as input_file:
        grant_result = pickle.load(input_file)
except:
        grant_result = {}

In [5]:
grant_result["time"] = dt.datetime.now()

In [6]:
with pymssql.connect(mssql['server'], mssql['user'], mssql['password'], mssql['database']) as conn:
    cursor = conn.cursor()
    select_sql = '''
                SELECT P.NameLang1, AddressLang1, VPO.*, TAC.*   FROM mars.[dbo].[TakeAwayPoiConfig] TAC (nolock)
                INNER JOIN mars.[dbo].[Poi] P  (nolock) ON P.Poiid = TAC.Poiid
                INNER JOIN mars.[dbo].[VendorPoiPosOperation] VPO (nolock) ON VPO.ORpoiid = P.Orpoiid
                INNER JOIN mars.dbo.BizService BS (nolock) ON BS.Poiid = TAC.Poiid
                INNER JOIN mars.dbo.BizContract BC (nolock) ON BC.ContractId = BS.ContractId
                WHERE BS.ServiceTypeId = 4 AND BS.[Status] = 10 AND BS.[ServiceEndTime] > GETDATE()
                AND (isEnableAcceptOrder = 0 OR TakeawayStatus = 0)
                AND BC.MetaData LIKE '%"IntegratedSystem":30%'
                AND BizVendorId = 28
                AND P.[Status] = 10
                UNION
                SELECT P.NameLang1, AddressLang1, VPO.*, TAC.*   FROM mars.[dbo].[TakeAwayPoiConfig] TAC (nolock)
                INNER JOIN mars.[dbo].[Poi] P (nolock) ON P.Poiid = TAC.Poiid
                INNER JOIN mars.[dbo].[VendorPoiPosOperation] VPO (nolock) ON VPO.ORpoiid = P.Orpoiid
                INNER JOIN mars.dbo.BizService BS (nolock) ON BS.Poiid = TAC.Poiid
                INNER JOIN mars.dbo.BizContract BC (nolock) ON BC.ContractId = BS.ContractId
                WHERE BS.ServiceTypeId = 4 AND BS.[Status] = 10 AND BS.[ServiceEndTime] > GETDATE()
                AND (isEnableAcceptOrder = 0 OR TakeawayStatus = 0)
                AND BC.MetaData NOT LIKE '%"IntegratedSystem":30%'
                AND BizVendorId = 2
                AND P.[Status] = 10    
                '''
    cursor.execute(select_sql)
    disabled_poi = cursor.fetchall()
    disabled_poi = [x[13] for x in disabled_poi] 

In [7]:
with pymssql.connect(mssql['server'], mssql['user'], mssql['password'], mssql['database']) as conn:
    cursor = conn.cursor()
    select_sql = '''
        select  p.poiid from openrice3.dbo.poi a
        left join mars.dbo.poi p on p.orpoiid=a.poiid
        where a.status=10 and a.regionid=0 
                '''
    cursor.execute(select_sql)
    active_poi = cursor.fetchall()
    active_poi = [x[0] for x in active_poi] 

In [8]:
len(active_poi)

50777

In [9]:
iso_to_poihourDate = {1:2,
                      2:3,
                      3:4,
                      4:5,
                      5:6,
                      6:7,
                      7:1,}

In [3]:
from datetime import datetime, timedelta

# yesterday = datetime.now() - timedelta(days=1)
yesterday = datetime(2026, 10, 2, 10, 0, 0)
iso_weekday = yesterday.isoweekday()

print(yesterday)
print(iso_weekday)

2026-10-02 10:00:00
5


In [10]:
with pymssql.connect(mssql['server'], mssql['user'], mssql['password'], mssql['database']) as conn:
    cursor = conn.cursor()
    select_sql = f'''
             SELECT  p.[PoiId] as poiid
              FROM [openrice3].[dbo].[PoiHour] (nolock) ph
              left join mars.dbo.poi (nolock) p on p.orpoiid= ph.poiid
              where isClose=1 and DayOfWeek is not null and p.status=10
              and dayofweek={iso_to_poihourDate[iso_weekday]}
                '''
    cursor.execute(select_sql)
    day_off_poi = cursor.fetchall()
    day_off_poi = [x[0] for x in day_off_poi] 

In [11]:
len(day_off_poi)

4266

In [8]:
# add one to poi if disabled
for i in disabled_poi:
    if i not in day_off_poi:
        if i in grant_result:
            grant_result[i]+=1
        else:
            grant_result[i]=1

In [9]:
# turn 0 if poi is on
on_poi = list(set(active_poi)-set(disabled_poi))
for i in on_poi:
    if i in grant_result:
        grant_result[i] = 0
    

In [10]:
# get inactive poi in grant_result
temp = list(grant_result.keys())
temp.remove("time")
inactive_poi_in_grant_result = list(set(temp)-set(active_poi))

In [11]:
# remove inactive poi from grant_result
for i in inactive_poi_in_grant_result:
    del grant_result[i]

In [12]:
#update pickle file
with open(r"offed_pois.obj", "wb") as output_file:
    pickle.dump(grant_result, output_file)

# prepare alert file

In [13]:
# alert_list=[]
# for i in grant_result:
#     if i == "time":
#         continue
#     if grant_result[i]>scope_day*24:
#         alert_list.append(i)

In [14]:
# alert_list_details=[]
# if len(alert_list)>0:
#     alert_list = tuple(alert_list)
#     if len(alert_list) == 1:
#         alert_list = str(alert_list)
#         alert_list = alert_list.replace(",","")
#     alert_list = str(alert_list)
#     with pymssql.connect(mssql['server'], mssql['user'], mssql['password'], mssql['database']) as conn:
#         cursor = conn.cursor()
#         select_sql = '''
#     select  ORPoiId,NameLang1,AddressLang1 from  mars.[dbo].[Poi] (nolock)
#     where poiid in {0}  
#                     '''.format(alert_list)
#         cursor.execute(select_sql)
#         alert_list_details = cursor.fetchall()

In [16]:
# _df = pd.DataFrame(alert_list_details,columns=["ORPoiId","NameLang1","AddressLang1"])

In [17]:
# filename = str(dt.datetime.now().date()) + '_' + str(scope_day) +'.xlsx'

In [18]:
# if len(alert_list_details)>0:
#     _df.to_excel(filename,index=False)
#     print("done")